# 从 PokemonDB 抓取宝可梦可学会的招式数据

本 notebook 会读取 `../data/move.csv` 和 `../data/pokemon151_v3.csv`，并从 `https://pokemondb.net/pokedex/[name_en]/moves/[latest_generation]` 抓取每只宝可梦的学习招式信息，导出为 `../data/learn.csv`。

In [9]:
import csv
import json
import re
import time
from pathlib import Path

try:
    import requests
    from bs4 import BeautifulSoup
except ImportError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'requests', 'beautifulsoup4'])
    import requests
    from bs4 import BeautifulSoup

BASE_URL = 'https://pokemondb.net'
POKEMON_CSV = Path('../data/pokemon151_v3.csv').resolve()
MOVE_CSV = Path('../data/move.csv').resolve()
OUTPUT_CSV = Path('../data/learn.csv').resolve()
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
}

session = requests.Session()
session.headers.update(HEADERS)

def slugify_pokemon_name(name: str):
    slug = name.strip().lower()
    slug = slug.replace('♀', '-f').replace('♂', '-m')
    slug = slug.replace('.', '').replace("'", '').replace('é', 'e')
    slug = re.sub(r'[^a-z0-9-]+', '-', slug)
    slug = re.sub(r'-{2,}', '-', slug).strip('-')
    return slug

def get_soup(url: str):
    response = session.get(url, timeout=20)
    response.raise_for_status()
    return BeautifulSoup(response.text, 'html.parser')

def load_csv(path: Path):
    with open(path, 'r', encoding='utf-8', newline='') as fp:
        return list(csv.DictReader(fp))

def build_move_lookup(moves):
    lookup = {}
    for row in moves:
        name = row.get('name_en', '').strip().lower()
        if not name:
            continue
        lookup[name] = row.get('move_id', '')
    return lookup

def normalize_method(title: str):
    text = (title or '').lower()
    if 'level up' in text or text == 'level up':
        return 'level_up'
    if 'egg' in text:
        return 'egg'
    if 'breeding' in text:
        return 'breeding'
    if 'hm' in text:
        return 'HM'
    if 'tm' in text and 'tr' in text:
        return 'TM'
    if 'tm' in text:
        return 'TM'
    if 'tr' in text:
        return 'TR'
    if 'tutor' in text:
        return 'tutor'
    if 'transfer-only' in text or 'transfer only' in text or 'transfer' in text:
        return 'transfer'
    if 'pre evolution' in text or 'pre-evolution' in text:
        return 'pre_evolution'
    if 'evolution' in text:
        return 'evolution'
    return 'transfer'

def parse_level(value: str):
    value = value.strip()
    if value.isdigit():
        return value
    match = re.search(r'(\d+)', value)
    return match.group(1) if match else ''

def find_next_table(heading):
    sibling = heading.next_sibling
    while sibling is not None:
        if getattr(sibling, 'name', None) == 'table':
            return sibling
        if getattr(sibling, 'name', None) == 'div':
            table = sibling.select_one('table.data-table')
            if table:
                return table
        sibling = sibling.next_sibling
    return None

def infer_method_transfer(text: str):
    text = (text or '').lower()
    if 'level' in text:
        return 'level_up'
    if 'egg' in text:
        return 'egg'
    if 'breed' in text:
        return 'breeding'
    if 'hm' in text:
        return 'HM'
    if 'tr' in text:
        return 'TR'
    if 'tm' in text:
        return 'TM'
    if 'tutor' in text:
        return 'tutor'
    if 'evol' in text:
        return 'evolution'
    return 'transfer'

def parse_transfer_gen(text: str, default_gen: str):
    if not text:
        return default_gen
    match = re.search(r'generation\s*(\d+)', text, re.I)
    if match:
        return match.group(1)
    return default_gen

def parse_learn_page(pokemon, move_lookup):
    slug = slugify_pokemon_name(pokemon['name_en'])
    latest_generation = pokemon.get('latest_generation', '').strip() or ''
    url = f'{BASE_URL}/pokedex/{slug}/moves/{latest_generation}'
    try:
        soup = get_soup(url)
    except requests.HTTPError:
        alternatives = [slug]
        if slug.endswith('-f'):
            alternatives.append(slug.replace('-f', 'f'))
        if slug.endswith('-m'):
            alternatives.append(slug.replace('-m', 'm'))
        soup = None
        for alt_slug in alternatives:
            alt_url = f'{BASE_URL}/pokedex/{alt_slug}/moves/{latest_generation}'
            try:
                soup = get_soup(alt_url)
                break
            except requests.HTTPError:
                continue
        if soup is None:
            print(f'Warning: 无法访问 {url} 或替代 slug 页面，跳过 {pokemon["name_en"]}')
            return []
    print(f'DEBUG: fetched {url} h2s={len(soup.select("h2"))} tables={len(soup.select("table.data-table"))}')
    learn_rows = []
    # prefer h2/h3/h4 headings (site often uses h3 for methods)
    headings = soup.select('h2,h3,h4')
    if headings:
        for heading in headings:
            title = heading.get_text(strip=True)
            method = normalize_method(title)
            table = find_next_table(heading)
            if table is None:
                continue
            # determine header columns and which index contains level (lv/Lv.)
            header_cells = [th.get_text(strip=True).lower() for th in table.select('thead th')]
            level_idx = None
            for i, h in enumerate(header_cells):
                if 'level' in h or h.replace('.', '').strip() in ('lv', 'lv'):
                    level_idx = i
                    break
            rows = table.select('tbody tr')
            # skip regional-form sections: if the nearest heading contains the pokemon name
            prev_heading = heading.get_text(strip=True).lower()
            pokemon_name_lower = pokemon['name_en'].strip().lower()
            if pokemon_name_lower in prev_heading and prev_heading != pokemon_name_lower:
                # heading mentions a form variant (e.g. 'galarian meowth'), skip
                continue
            for row in rows:
                cells = row.select('td')
                if not cells:
                    continue
                name_link = row.select_one('a.ent-name')
                if not name_link:
                    continue
                move_name_en = name_link.get_text(strip=True)
                move_id = move_lookup.get(move_name_en.strip().lower(), '')
                row_text = ' '.join(td.get_text(' ', strip=True) for td in cells)
                row_method = method
                method_transfer = ''
                learn_level = ''
                gen = latest_generation
                # for transfer sections, try to infer transfer subtype from heading+row text
                if method == 'transfer':
                    method_transfer = infer_method_transfer(f"{title} {row_text}")
                    gen = parse_transfer_gen(f"{title} {row_text}", latest_generation)
                # TM/TR markers may appear in the first cell
                if method in ('TM', 'TR') or method == 'transfer' and header_cells and header_cells[0].lower().startswith(('tm', 'tr')):
                    marker = cells[0].get_text(strip=True).lower()
                    if 'tm' in marker:
                        row_method = 'TM'
                    elif 'tr' in marker:
                        row_method = 'TR'
                    else:
                        row_method = method
                # level extraction: use identified index when possible
                if method == 'level_up':
                    if level_idx is not None and level_idx < len(cells):
                        learn_level = parse_level(cells[level_idx].get_text(strip=True))
                    else:
                        # fallback to first cell
                        learn_level = parse_level(cells[0].get_text(strip=True))
                elif method == 'egg':
                    learn_level = '1'
                elif method in ['breeding', 'HM', 'TM', 'TR', 'tutor']:
                    learn_level = '0'
                elif method == 'evolution':
                    learn_level = str(pokemon.get('evolution_level', '')).strip()
                elif method == 'pre_evolution':
                    learn_level = '0'
                elif method == 'transfer':
                    learn_level = '0'
                # ensure evolution/pre-evolution learn_level is numeric
                if method in ('evolution', 'pre_evolution') and not (learn_level or '').isdigit():
                    learn_level = '0'
                if method == 'transfer' and not method_transfer:
                    method_transfer = 'transfer'
                learn_rows.append({
                    'pokemon_id': pokemon['pokemon_id'],
                    'pokemon_name_en': pokemon['name_en'],
                    'move_name_en': move_name_en,
                    'move_id': move_id,
                    'gen': gen,
                    'method': row_method,
                    'method_transfer': method_transfer,
                    'learn_level': learn_level,
                })
    else:
        # fallback: iterate over all tables and infer method from previous heading (h2/h3/h4)
        for table in soup.select('table.data-table'):
            prev_heading = table.find_previous(lambda tag: tag.name in ['h2','h3','h4'])
            title = prev_heading.get_text(strip=True) if prev_heading else ''
            method = normalize_method(title)
            header_cells = [th.get_text(strip=True).lower() for th in table.select('thead th')]
            level_idx = None
            for i, h in enumerate(header_cells):
                if 'level' in h or h.replace('.', '').strip() in ('lv', 'lv'):
                    level_idx = i
                    break
            # skip regional-form sections similar to above
            prev_text = title.lower()
            pokemon_name_lower = pokemon['name_en'].strip().lower()
            if pokemon_name_lower in prev_text and prev_text != pokemon_name_lower:
                continue
            rows = table.select('tbody tr')
            for row in rows:
                cells = row.select('td')
                if not cells:
                    continue
                name_link = row.select_one('a.ent-name')
                if not name_link:
                    continue
                move_name_en = name_link.get_text(strip=True)
                move_id = move_lookup.get(move_name_en.strip().lower(), '')
                row_text = ' '.join(td.get_text(' ', strip=True) for td in cells)
                row_method = method
                method_transfer = ''
                learn_level = ''
                gen = latest_generation
                if method == 'transfer':
                    method_transfer = infer_method_transfer(f"{title} {row_text}")
                    gen = parse_transfer_gen(f"{title} {row_text}", latest_generation)
                if method in ('TM', 'TR') or method == 'transfer' and header_cells and header_cells[0].lower().startswith(('tm', 'tr')):
                    marker = cells[0].get_text(strip=True).lower()
                    if 'tm' in marker:
                        row_method = 'TM'
                    elif 'tr' in marker:
                        row_method = 'TR'
                    else:
                        row_method = method
                if method == 'level_up':
                    if level_idx is not None and level_idx < len(cells):
                        learn_level = parse_level(cells[level_idx].get_text(strip=True))
                    else:
                        learn_level = parse_level(cells[0].get_text(strip=True))
                elif method == 'egg':
                    learn_level = '1'
                elif method in ['breeding', 'HM', 'TM', 'TR', 'tutor']:
                    learn_level = '0'
                elif method == 'evolution':
                    learn_level = str(pokemon.get('evolution_level', '')).strip()
                elif method == 'pre_evolution':
                    learn_level = '0'
                elif method == 'transfer':
                    learn_level = '0'
                if method in ('evolution', 'pre_evolution') and not (learn_level or '').isdigit():
                    learn_level = '0'
                if method == 'transfer' and not method_transfer:
                    method_transfer = 'transfer'
                learn_rows.append({
                    'pokemon_id': pokemon['pokemon_id'],
                    'pokemon_name_en': pokemon['name_en'],
                    'move_name_en': move_name_en,
                    'move_id': move_id,
                    'gen': gen,
                    'method': row_method,
                    'method_transfer': method_transfer,
                    'learn_level': learn_level,
                })
    return learn_rows

def save_csv(learn_data):
    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = [
        'learn_id',
        'pokemon_id',
        'move_id',
        'pokemon_name_en',
        'move_name_en',
        'gen',
        'method',
        'method_transfer',
        'learn_level',
    ]
    with open(OUTPUT_CSV, 'w', encoding='utf-8', newline='') as fp:
        writer = csv.DictWriter(fp, fieldnames=fieldnames)
        writer.writeheader()
        for row in learn_data:
            writer.writerow({key: row.get(key, '') for key in fieldnames})

def main():
    pokemon_rows = load_csv(POKEMON_CSV)
    move_rows = load_csv(MOVE_CSV)
    move_lookup = build_move_lookup(move_rows)
    all_learn = []
    learn_id = 1
    for pokemon in pokemon_rows:
        learn_entries = parse_learn_page(pokemon, move_lookup)
        for entry in learn_entries:
            entry['learn_id'] = learn_id
            all_learn.append(entry)
            learn_id += 1
            time.sleep(0.2)
    save_csv(all_learn)
    print(f'完成：已保存 {len(all_learn)} 条学习招式数据到 {OUTPUT_CSV}')

main()

DEBUG: fetched https://pokemondb.net/pokedex/bulbasaur/moves/9 h2s=0 tables=5


DEBUG: fetched https://pokemondb.net/pokedex/bulbasaur/moves/9 h2s=0 tables=5


KeyboardInterrupt: 